In [1]:
VERSION = "1.1.0"

In [2]:
from pathlib import Path

import tensorflow_datasets
from astropy.io import fits
from jax import numpy
from pandas import DataFrame
from polars import read_csv
from tqdm import tqdm
from utils import search_folder


def write_fits(array, path):
    hdu = fits.PrimaryHDU(array)
    hdulist = fits.HDUList([hdu])
    hdulist.writeto(path, overwrite=True)
    hdulist.close()


DIR_DATASETS = search_folder(name="DATASETS")
DATASET_VERSION = DIR_DATASETS / "pisco_euclid" / f"{VERSION}_sextractor"
DIR_SEXTRACTOR = Path(f"{DATASET_VERSION}")
DIR_SEXTRACTOR.mkdir(parents=True, exist_ok=True)

2025-02-20 17:32:20.633727: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-20 17:32:20.667868: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
used_dataset = tensorflow_datasets.load(
    f"pisco_euclid:{VERSION}", split="train", data_dir=DIR_DATASETS, shuffle_files=False
)

print(TOTAL_ELEMENTS := used_dataset.cardinality().numpy())

all_params_dataset = used_dataset.map(
    lambda x: {
        "avg_e": x["avg_e"],
        "avg_g": x["avg_g"],
        "image": x["image"],
        "gal_params": x["gal_params"],
    }
)

all_params = list(all_params_dataset.as_numpy_iterator())
ellips = numpy.array([item["avg_e"] for item in all_params])
shears = numpy.array([item["avg_g"] for item in all_params])
images = numpy.array([item["image"] for item in all_params])
gal_params = numpy.array([item["gal_params"] for item in all_params])

del used_dataset, all_params_dataset, all_params

4000


In [4]:
columns = [
    "ell_1",
    "ell_2",
    "x",
    "y",
    "offset_x",
    "offset_y",
    "mag",
    "half_light_radius",
    "n_sersic",
]

for sample in tqdm(range(TOTAL_ELEMENTS)):
    write_fits(images[sample], DIR_SEXTRACTOR / f"simu_{sample + 1:05d}.fits")
    DataFrame(gal_params[sample].T, columns=columns).to_csv(
        DIR_SEXTRACTOR / f"true_galaxies_params_{sample + 1:05d}.csv", index=False
    )
    # write_fits(numpy.array(shears[sample]), DIR_SEXTRACTOR / f"avg_shear_{sample + 1:05d}.fits")
    # write_fits(numpy.array(ellips[sample]), DIR_SEXTRACTOR / f"avg_ellips_{sample + 1:05d}.fits")

DataFrame(
    numpy.vstack([shears[:, 0], shears[:, 1], ellips[:, 0], ellips[:, 1]]).T,
    columns=["avg_g1", "avg_g2", "avg_e1", "avg_e2"],
).to_csv(DIR_SEXTRACTOR / "true_images_params.csv", index=False)

100%|██████████| 4000/4000 [00:35<00:00, 114.08it/s]


## test

In [ ]:
# # with fits.open(DIR_SEXTRACTOR / "simu_00010.fits") as hdul:
# with fits.open(DIR_SEXTRACTOR / "avg_ellips_00001.fits") as hdul:
#     # with fits.open(DIR_SEXTRACTOR / "avg_shear_00000.fits") as hdul:

#     data = hdul[0].data

# print(data.shape)
# print(data)

In [ ]:
true_images_params = read_csv(DATASET_VERSION / "true_images_params.csv", skip_rows=0)
true_images_params

In [ ]:
DataFrame(
    numpy.vstack([shears[:, 0], shears[:, 1], ellips[:, 0], ellips[:, 1]]).T,
    columns=["avg_g1", "avg_g2", "avg_e1", "avg_e2"],
)